In [15]:
# !pip install    

In [16]:
import pandas as pd

In [17]:
df = pd.read_csv('data/Dataset_English_Hindi.csv')

In [18]:
df.shape

(130476, 2)

In [19]:
df.isnull().sum()

English      2
Hindi      312
dtype: int64

In [20]:
df = df.dropna(subset=['English', 'Hindi']).reset_index(drop=True)

In [21]:
df.head()

,English,Hindi
0,Help!,बचाओ!
1,Jump.,उछलो.
2,Jump.,कूदो.
3,Jump.,छलांग.
4,Hello!,नमस्ते।


In [22]:
df= df.sample(n=5000, random_state=42).reset_index(drop=True)
df.shape

(5000, 2)

In [23]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print("Train:", train_data.shape)
print("Validation:", val_data.shape)

Train: (4000, 2)
Validation: (1000, 2)


In [24]:
train_data.shape, val_data.shape

((4000, 2), (1000, 2))

In [25]:
train_data.head()

,English,Hindi
0,Innings,पारियां
1,"When I was 10 years old,","जब मैं १० वर्ष कि थी,"
2,What was of more significance was that the Ind...,अधिक महत्व की बात यह थी कि भारतीय मिलें अब घरे...
3,how big these collections are or how big the i...,और ये संग्रह कितना बड़ा है या ये तस्वीरें कितन...
4,The majority of nurseries are open weekdays al...,अधिकांश नर्सरीज़कार्य दिवसों पर साल भर खुली रह...


In [26]:
def clean_english(text):
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    return text.strip().lower()


def clean_hindi(text):
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    return text.strip()

In [27]:
train_data.head()

,English,Hindi
0,Innings,पारियां
1,"When I was 10 years old,","जब मैं १० वर्ष कि थी,"
2,What was of more significance was that the Ind...,अधिक महत्व की बात यह थी कि भारतीय मिलें अब घरे...
3,how big these collections are or how big the i...,और ये संग्रह कितना बड़ा है या ये तस्वीरें कितन...
4,The majority of nurseries are open weekdays al...,अधिकांश नर्सरीज़कार्य दिवसों पर साल भर खुली रह...


In [28]:
from torch.utils.data import Dataset, DataLoader

class TranslatorDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer


    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        # english = self.data.iloc[index]['English']
        english = str(self.data.iloc[index]['English'])
        hindi = str(self.data.iloc[index]['Hindi'])

        # T5 task instruction
        source_text = "translate English to Hindi: " + english

        source = self.tokenizer(source_text, padding='max_length', max_length=128, truncation=True, return_tensors='pt')
        target = self.tokenizer(text_target = hindi, padding='max_length', max_length=128, truncation=True, return_tensors='pt')

        labels = target['input_ids'].squeeze(0)

        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            'input_ids':source['input_ids'].squeeze(0),
            'attention_mask':source['attention_mask'].squeeze(0),
            'labels':labels
        }
        
        
        

In [30]:
from transformers import MarianTokenizer, MarianMTModel

model_name = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

/Users/hemantpatidar/miniconda3/envs/translator/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [31]:
train_dataset = TranslatorDataset(train_data, tokenizer)
val_dataset = TranslatorDataset(val_data, tokenizer)

In [32]:
sample = train_dataset[0]

print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["labels"].shape)

torch.Size([128])
torch.Size([128])
torch.Size([128])


In [33]:
print("English:")
print(train_data.iloc[0]["English"])

print("\nHindi:")
print(train_data.iloc[0]["Hindi"])

English:
Innings

Hindi:
पारियां


In [34]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [35]:
len(train_loader), len(val_loader)

(500, 125)

In [36]:
import torch

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

print(device)

mps


In [37]:

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

model.to(device)
print("Device:", device)
print("Model device:", next(model.parameters()).device)

Device: mps
Model device: mps:0


In [38]:
epochs = 4

for epoch in range(epochs):
    model.train()

    total_loss = 0.0

    for batch in train_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = output.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss/len(train_loader)

    print(f"epoch {epoch+1}/{epochs} training loss {avg_loss}")
   
        

epoch 1/4 training loss 4.239627370357513
epoch 2/4 training loss 3.1384867446422575
epoch 3/4 training loss 2.3733649604320526
epoch 4/4 training loss 1.7578193817138672


In [39]:
model.eval()
val_loss = 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        output = model(
                input_ids = input_ids,
                attention_mask = attention_mask,
                labels= labels
            )

        loss = output.loss

        val_loss += loss.item()

    avg_loss = val_loss/len(val_loader)

    print(f"epoch {epoch+1}/{epochs}  test loss {avg_loss}")

        

epoch 4/4  test loss 4.238587797164917


In [40]:
model.eval()

def translate_english_to_hindi(text):
    source_text = text

    inputs = tokenizer(
        source_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_length=128,
            num_beams=5,
            early_stopping=True
        )

    translation = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    return translation

In [42]:
test_sentences = [
    "How are you?",
    "What is your name?",
    "I am learning machine learning.",
    "I live in India.",
    "Today is a beautiful day.",
    "I want to become an AI engineer.",
    "Where are you going?",
    "Please give me some water.",
    "I have completed my project.",
    "The weather is very nice today."
]

for text in test_sentences:
    print("English :", text)
    print("Hindi   :", translate_english_to_hindi(text))
    print("-" * 60)

English : How are you?
Hindi   : तुम कैसे हो?
------------------------------------------------------------
English : What is your name?
Hindi   : आपका नाम क्या है?
------------------------------------------------------------
English : I am learning machine learning.
Hindi   : मैं मशीन सीखने जा रहा हूँ।
------------------------------------------------------------
English : I live in India.
Hindi   : मैं भारत में रहता हूँ।
------------------------------------------------------------
English : Today is a beautiful day.
Hindi   : आज एक भव्य दिन है.
------------------------------------------------------------
English : I want to become an AI engineer.
Hindi   : मैं एक %s इंजीनियर बनना चाहती हूँ।
------------------------------------------------------------
English : Where are you going?
Hindi   : तुम कहाँ जा रहे हो?
------------------------------------------------------------
English : Please give me some water.
Hindi   : मुझे पानी दो।
--------------------------------------------------------